In [35]:
# PyTorch 7

# Training model on GPU
# Full training fMNIST dataset (60000 images)

In [36]:
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchmetrics
from torch.utils.data import Dataset,DataLoader

In [37]:
# kaggle link : "https://www.kaggle.com/datasets/zalando-research/fashionmnist?resource=download"

# !pip install torchmetrics

In [38]:
df = pd.read_csv("/content/drive/MyDrive/fmnist_full_dataset/fashion-mnist_train.csv")

df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [39]:
df.shape

(60000, 785)

In [40]:
# checking for GPU availability

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device

device(type='cuda')

In [41]:
# feats & labels

X = df.drop('label',axis=1).values
y = df['label'].values

In [42]:
# train-test split

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=38)

In [43]:
X_train.shape

(48000, 784)

In [44]:
# scaling

X_train = X_train/255.0
X_test = X_test/255.0

In [45]:
# CustomDataset

class CustomDataset(Dataset):

  def __init__(self,x,y):
    self.x = torch.tensor(x).float()
    self.y = torch.tensor(y)

  def __len__(self):
    return len(self.x)

  def __getitem__(self, idx):
    return self.x[idx],self.y[idx]

In [46]:
# instances of CustomDataset

train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

In [47]:
# DataLoader with pin_memory = True
# skips data loading to page_memory in memory thus saving time

train_dataloader = DataLoader(train_dataset,batch_size=64,shuffle=True,pin_memory=True)
test_dataloader = DataLoader(test_dataset,batch_size=64,shuffle=False,pin_memory=True)

In [48]:
# Network building

class NeuNet(nn.Module):
    def __init__(self, input_size):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
      out = self.network(x)
      return out

In [49]:
# instance of NeuNet

model = NeuNet(X_train.shape[1])

# move to GPU

model = model.to(device)

In [50]:
learning_rate = 0.001

epochs = 100

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(),lr=learning_rate)

In [51]:
# training pipeline

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_feats,batch_labels in train_dataloader:

    batch_feats = batch_feats.to(device)                     # moving to GPU
    batch_labels = batch_labels.to(device)                   # moving to GPU

    outputs = model(batch_feats)

    loss = criterion(outputs,batch_labels)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    total_epoch_loss += loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_epoch_loss/len(train_dataloader)}")

Epoch: 1, Loss: 0.5994651011029879
Epoch: 2, Loss: 0.41551863634586333
Epoch: 3, Loss: 0.3726452036301295
Epoch: 4, Loss: 0.344639308253924
Epoch: 5, Loss: 0.32098540516694385
Epoch: 6, Loss: 0.3051945189833641
Epoch: 7, Loss: 0.29199634128808977
Epoch: 8, Loss: 0.2801832577586174
Epoch: 9, Loss: 0.2678944548765818
Epoch: 10, Loss: 0.25811037360628447
Epoch: 11, Loss: 0.24842461147904396
Epoch: 12, Loss: 0.23915094556411107
Epoch: 13, Loss: 0.23260748915870985
Epoch: 14, Loss: 0.2241196934382121
Epoch: 15, Loss: 0.2190459387054046
Epoch: 16, Loss: 0.21111277819673221
Epoch: 17, Loss: 0.2042309121787548
Epoch: 18, Loss: 0.20018800140420595
Epoch: 19, Loss: 0.1896460314144691
Epoch: 20, Loss: 0.18767855687439441
Epoch: 21, Loss: 0.17994368465741475
Epoch: 22, Loss: 0.17696355012059212
Epoch: 23, Loss: 0.17432765122999747
Epoch: 24, Loss: 0.16384794103850922
Epoch: 25, Loss: 0.16140460978945095
Epoch: 26, Loss: 0.15476595591505368
Epoch: 27, Loss: 0.15332655203342438
Epoch: 28, Loss: 0.14

In [52]:
from torchmetrics.classification import MulticlassAccuracy

metrics = MulticlassAccuracy(num_classes=10).to(device)

In [53]:
# model evaluation

model.eval()

with torch.no_grad():

    for batch_feats, batch_labels in test_dataloader:

        batch_feats = batch_feats.to(device)
        batch_labels = batch_labels.to(device)

        y_pred = model(batch_feats)

        metrics.update(y_pred, batch_labels)

total_accuracy = metrics.compute()
print(f"Test Accuracy: {total_accuracy:.2f}")

metrics.reset()

Test Accuracy: 0.88
